[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/05_numerical_differentiation/first_principles.ipynb)

# Topic 05: Numerical Differentiation

## 1. First-Principles Intuition & Motivation

Differentiation is the easiest operation in calculus and the hardest in floating point. The definition

$$
f'(x) = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h}
$$

suggests an algorithm so obvious it barely deserves the name: pick a small $h$ and evaluate the quotient. Try it in double precision on $f = \sin$ at $x = 1$ and the error behaves as follows: $4.2\times10^{-2}$ at $h = 10^{-1}$, improving steadily to $3.0\times10^{-9}$ at $h = 10^{-8}$ — and then getting **worse**, reaching $1.5\times10^{-2}$ at $h = 10^{-15}$ and $0.54$ (i.e. $100\%$ error) at $h = 10^{-16}$.

The limit does not exist numerically. Two error mechanisms pull in opposite directions:

- **Truncation error**: the difference quotient is not the derivative; Taylor's theorem says it differs by $\frac{h}{2}f''(\xi)$, which *decreases* with $h$.
- **Roundoff error**: $f(x+h)$ and $f(x)$ are each computed with a relative error $\approx \varepsilon_M$; as $h \to 0$ the two values agree in more and more leading digits, so their difference retains fewer and fewer correct digits — **catastrophic cancellation** — and then that corrupted difference is amplified by dividing by a tiny $h$. The roundoff contribution *increases* like $\varepsilon_M/h$.

### The V-shaped error curve

Adding the two mechanisms gives the total-error model

$$
E(h) \approx \underbrace{C_t\,h^{p}}_{\text{truncation}} + \underbrace{\frac{C_r\,\varepsilon_M}{h}}_{\text{roundoff}} ,
$$

whose graph on log–log axes is a **V**: slope $+p$ on the left (truncation dominant), slope $-1$ on the right (roundoff dominant), with a minimum at a finite $h^{\ast}$. Minimizing gives

$$
h^{\ast} = \left( \frac{C_r \varepsilon_M}{p\,C_t} \right)^{1/(p+1)}, \qquad E(h^{\ast}) = O\!\left( \varepsilon_M^{\,p/(p+1)} \right).
$$

For the first-order forward difference ($p = 1$): $h^{\ast} \sim \sqrt{\varepsilon_M} \approx 1.5\times10^{-8}$ and best accuracy $\sim \sqrt{\varepsilon_M} \approx 10^{-8}$ — **half the digits are gone, permanently**. For the second-order central difference ($p = 2$): $h^{\ast} \sim \varepsilon_M^{1/3} \approx 6\times10^{-6}$ and best accuracy $\sim \varepsilon_M^{2/3} \approx 4\times10^{-11}$. For the second-derivative stencil ($p = 2$ but roundoff $\sim \varepsilon_M/h^2$): $h^{\ast} \sim \varepsilon_M^{1/4} \approx 1.2\times10^{-4}$ and accuracy only $\sim \sqrt{\varepsilon_M}$.

This is not a defect of any particular implementation. Numerical differentiation is an **ill-posed problem**: an $O(\eta)$ perturbation of $f$ (add $\eta \sin(x/\eta^2)$) changes $f'$ by $O(1/\eta)$ while changing $f$ by $O(\eta)$. Differentiation *amplifies* noise; integration (Topic 06) *averages* it away. Every difficulty in this topic follows from that asymmetry.

Three escapes exist, and the rest of this notebook develops them: raise the order (better stencils, Richardson extrapolation), remove the subtraction (complex-step differentiation), or abandon approximation entirely (automatic differentiation).

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Finite-difference operators).** For step $h \gt 0$,

$$
\Delta_h f(x) = f(x+h) - f(x) \ \text{(forward)}, \quad \nabla_h f(x) = f(x) - f(x-h) \ \text{(backward)}, \quad \delta_h f(x) = f\!\left(x + \tfrac h2\right) - f\!\left(x - \tfrac h2\right) \ \text{(central)} .
$$

**Definition 2 (Difference formula and order of accuracy).** A formula $D_h f(x) = \frac{1}{h^{k}}\sum_{j} c_j\, f(x + a_j h)$ approximates $f^{(k)}(x)$ with **order of accuracy $p$** if

$$
D_h f(x) - f^{(k)}(x) = O(h^{p}) \quad \text{as } h \to 0
$$

for all sufficiently smooth $f$. The set $\{a_j\}$ is the **stencil** and $\{c_j\}$ the **weights**.

**Definition 3 (Consistency conditions).** $D_h$ has order $\ge p$ iff it is exact on all polynomials of degree $\le k + p - 1$, equivalently iff

$$
\sum_j c_j \frac{a_j^{\,i}}{i!} = \delta_{i,k}, \qquad i = 0, 1, \ldots, k+p-1 .
$$

This is a (transposed) Vandermonde system in the weights $c_j$ — always solvable for distinct $a_j$, which is why a stencil of $m$ points yields order $m - k$ generically.

**Definition 4 (Machine epsilon and evaluation noise).** In IEEE-754 double precision $\varepsilon_M = 2^{-53} \approx 1.11\times10^{-16}$ (unit roundoff), and the computed value satisfies $\mathrm{fl}(f(x)) = f(x)(1 + \theta)$ with $\lvert \theta \rvert \le \eta$, where $\eta \ge \varepsilon_M$ is the *evaluation noise level* — larger if $f$ is itself the output of an iterative solver or a simulation.

**Definition 5 (Richardson extrapolation).** Given a family $A(h)$ with the asymptotic expansion $A(h) = A + c_1 h^{p} + c_2 h^{p+q} + \cdots$, the extrapolant is

$$
R(h) = \frac{2^{p} A(h/2) - A(h)}{2^{p} - 1} = A + O\!\left(h^{p+q}\right).
$$

**Definition 6 (Complex-step derivative).** For $f$ real-analytic near $x$ with a complex-differentiable implementation,

$$
D^{\mathrm{CS}}_h f(x) = \frac{\operatorname{Im} f(x + ih)}{h}.
$$

**Definition 7 (Condition number of differentiation).** Relative to sup-norm perturbations of $f$ on an interval, differentiation is *unbounded*: for any $\eta \gt 0$ there is $g$ with $\lVert g \rVert_\infty \le \eta$ and $\lVert g' \rVert_\infty$ arbitrarily large. Restricted to a fixed step $h$, the amplification factor of the difference quotient is $2/h$.

**Theorem 1 (Forward and backward difference).** For $f \in C^{2}$ near $x$ there is $\xi$ between $x$ and $x+h$ with

$$
\frac{f(x+h) - f(x)}{h} = f'(x) + \frac{h}{2} f''(\xi) ,
$$

so the order of accuracy is $p = 1$. The backward formula is identical with $h \mapsto -h$.

**Theorem 2 (Central difference).** For $f \in C^{3}$ there is $\xi \in (x-h, x+h)$ with

$$
\frac{f(x+h) - f(x-h)}{2h} = f'(x) + \frac{h^{2}}{6} f'''(\xi) ,
$$

so $p = 2$. The error expansion contains **only even powers**: $\frac{h^2}{6}f''' + \frac{h^4}{120}f^{(5)} + \cdots$.

**Theorem 3 (Second derivative, three-point central).** For $f \in C^{4}$,

$$
\frac{f(x+h) - 2f(x) + f(x-h)}{h^{2}} = f''(x) + \frac{h^{2}}{12} f^{(4)}(\xi) .
$$

**Theorem 4 (Five-point first derivative).** For $f \in C^{5}$,

$$
\frac{f(x-2h) - 8f(x-h) + 8f(x+h) - f(x+2h)}{12h} = f'(x) - \frac{h^{4}}{30} f^{(5)}(\xi) .
$$

**Theorem 5 (Optimal step size).** Let $f$ be evaluated with absolute noise $\le \eta\,\lvert f \rvert$. Then

- **Forward difference**: total error $\le \frac{h}{2}M_2 + \frac{2\eta \lvert f \rvert}{h}$, minimized at
$$
h^{\ast} = 2\sqrt{\frac{\eta \lvert f \rvert}{M_2}} \sim \sqrt{\varepsilon_M}, \qquad E(h^{\ast}) = 2\sqrt{\eta \lvert f \rvert M_2} \sim \sqrt{\varepsilon_M}.
$$
- **Central difference**: total error $\le \frac{h^2}{6}M_3 + \frac{\eta \lvert f \rvert}{h}$, minimized at
$$
h^{\ast} = \left( \frac{3\eta \lvert f \rvert}{M_3} \right)^{1/3} \sim \varepsilon_M^{1/3}, \qquad E(h^{\ast}) \sim \varepsilon_M^{2/3}.
$$
- **Second derivative**: $h^{\ast} \sim \varepsilon_M^{1/4}$, $E(h^{\ast}) \sim \sqrt{\varepsilon_M}$.

Here $M_k = \max \lvert f^{(k)} \rvert$ on a neighbourhood of $x$.

**Theorem 6 (Richardson for central differences).** With $D_h = \frac{f(x+h)-f(x-h)}{2h}$, the even-power expansion gives

$$
\frac{4 D_{h/2} - D_h}{3} = f'(x) + O(h^{4}),
$$

and iterating produces a Romberg-style table with orders $2, 4, 6, \ldots$

**Theorem 7 (Complex step).** If $f$ is real-analytic near $x$ and real-valued on the real axis, then

$$
\frac{\operatorname{Im} f(x + ih)}{h} = f'(x) - \frac{h^{2}}{6}f'''(x) + O(h^{4}),
$$

with **no subtraction of nearly equal quantities**, so $h$ may be taken as small as $10^{-20}$ and the result is accurate to machine precision.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1 — Forward difference: order 1

**Claim.** (Theorem 1.) $\frac{f(x+h)-f(x)}{h} = f'(x) + \frac{h}{2}f''(\xi)$.

**Proof.** Taylor's theorem with Lagrange remainder, expanding about $x$ to second order: for $f \in C^2$ there is $\xi \in (x, x+h)$ with

$$
f(x + h) = f(x) + h f'(x) + \frac{h^{2}}{2} f''(\xi).
$$

Subtract $f(x)$ and divide by $h$:

$$
\frac{f(x+h) - f(x)}{h} = f'(x) + \frac{h}{2}f''(\xi). \qquad \blacksquare
$$

**Sharpness.** The bound $\lvert E_t \rvert \le \frac{h}{2}M_2$ is attained for $f(x) = x^2$, where $f'' \equiv 2$ and the forward quotient at $x$ equals $2x + h = f'(x) + h$, exactly $\frac{h}{2}\cdot 2$. So the constant $\tfrac12$ cannot be improved.

**Why only order 1.** The formula uses two function values and must reproduce $f$ and $f'$; two conditions on two weights leaves nothing left over. Symmetry is what buys the extra order in the central formula — see Proof 2.

### Proof 2 — Central differences: order 2 and even-power expansion

**Claim.** (Theorem 2.) $\frac{f(x+h)-f(x-h)}{2h} = f'(x) + \frac{h^2}{6}f'''(\xi)$, and the full expansion has only even powers of $h$.

**Proof.** Expand in both directions to third order, with remainders at $\xi_+ \in (x, x+h)$ and $\xi_- \in (x-h, x)$:

$$
f(x+h) = f(x) + h f'(x) + \frac{h^2}{2}f''(x) + \frac{h^3}{6}f'''(\xi_+),
$$

$$
f(x-h) = f(x) - h f'(x) + \frac{h^2}{2}f''(x) - \frac{h^3}{6}f'''(\xi_-).
$$

Subtracting, the $f(x)$ and $\frac{h^2}{2}f''(x)$ terms **cancel identically** — this is the payoff of symmetry:

$$
f(x+h) - f(x-h) = 2h f'(x) + \frac{h^{3}}{6}\bigl( f'''(\xi_+) + f'''(\xi_-) \bigr).
$$

Dividing by $2h$ and applying the intermediate value theorem to $f'''$ (continuous, so it attains the average of two of its values at some $\xi$):

$$
\frac{f(x+h)-f(x-h)}{2h} = f'(x) + \frac{h^2}{12}\bigl(f'''(\xi_+)+f'''(\xi_-)\bigr) = f'(x) + \frac{h^{2}}{6}f'''(\xi). \qquad \blacksquare
$$

**Even-power structure.** For $f \in C^\infty$, write the infinite expansions. The terms of even order in $h$ ($f(x)$, $\frac{h^2}{2}f''$, $\frac{h^4}{24}f^{(4)}$, …) appear with the *same* sign in both expansions and cancel on subtraction; the odd-order terms double. Hence

$$
\frac{f(x+h) - f(x-h)}{2h} = f'(x) + \frac{h^{2}}{6}f'''(x) + \frac{h^{4}}{120}f^{(5)}(x) + \cdots ,
$$

an expansion in $h^2$ alone. **This is exactly the hypothesis Richardson extrapolation needs** ($p = 2$, $q = 2$), and it is why one extrapolation step jumps from order 2 straight to order 4 rather than to order 3.

**Second derivative (Theorem 3).** Now *add* the two expansions, carried to fourth order. The odd terms cancel and

$$
f(x+h) + f(x-h) = 2f(x) + h^2 f''(x) + \frac{h^4}{12}f^{(4)}(\xi),
$$

giving $\frac{f(x+h)-2f(x)+f(x-h)}{h^2} = f''(x) + \frac{h^2}{12}f^{(4)}(\xi)$. $\blacksquare$

### Proof 3 — The roundoff–truncation trade-off and the optimal step

**Setup.** Suppose the computed values satisfy $\tilde f(x \pm h) = f(x \pm h) + e_{\pm}$ with $\lvert e_\pm \rvert \le \eta \lvert f \rvert$, where $\eta \ge \varepsilon_M$ (in the best case $\eta = \varepsilon_M$; for a noisy simulator it can be $10^{-6}$ or worse).

**Forward difference.** The computed quotient is

$$
\tilde D_h = \frac{\tilde f(x+h) - \tilde f(x)}{h} = \underbrace{\frac{f(x+h)-f(x)}{h}}_{= f'(x) + \frac{h}{2}f''(\xi)} + \frac{e_+ - e_0}{h},
$$

so the total error obeys

$$
E(h) \le \underbrace{\frac{h}{2}M_2}_{\text{truncation}} + \underbrace{\frac{2\eta \lvert f \rvert}{h}}_{\text{roundoff}} .
$$

Differentiate with respect to $h$ and set to zero:

$$
\frac{M_2}{2} - \frac{2\eta \lvert f \rvert}{h^{2}} = 0 \implies h^{\ast} = 2\sqrt{\frac{\eta \lvert f \rvert}{M_2}}, \qquad E(h^{\ast}) = 2\sqrt{\eta \lvert f \rvert M_2}.
$$

Both scale like $\sqrt{\eta}$: with $\eta = \varepsilon_M$ and $O(1)$ derivatives, $h^{\ast} \approx 3\times10^{-8}$ and $E \approx 3\times10^{-8}$. **The best achievable accuracy is the square root of machine precision — about 8 of 16 digits.**

**Central difference.** Same argument with $E(h) \le \frac{h^{2}}{6}M_3 + \frac{\eta \lvert f \rvert}{h}$:

$$
\frac{h M_3}{3} - \frac{\eta \lvert f \rvert}{h^{2}} = 0 \implies h^{\ast} = \left( \frac{3\eta \lvert f \rvert}{M_3} \right)^{1/3}, \qquad E(h^{\ast}) = \frac{3^{2/3}}{2}\,M_3^{1/3}\bigl(\eta \lvert f \rvert\bigr)^{2/3} \ \sim\ \eta^{2/3},
$$

using $\frac{3^{2/3}}{6} + 3^{-1/3} = \frac{3^{2/3}}{2}$. With $\eta = \varepsilon_M$ and $O(1)$ derivatives: $h^{\ast} \approx 10^{-5}$ and $E \approx 3\times10^{-11}$ — about $11$ correct digits.

**Second derivative.** Roundoff now enters as $\frac{4\eta \lvert f \rvert}{h^{2}}$ (four values, denominator $h^2$), so $E(h) \le \frac{h^2}{12}M_4 + \frac{4\eta\lvert f\rvert}{h^2}$ and

$$
h^{\ast} = \left( \frac{48\,\eta \lvert f \rvert}{M_4} \right)^{1/4} \sim \varepsilon_M^{1/4} \approx 1.2\times10^{-4}, \qquad E(h^{\ast}) \sim \sqrt{\varepsilon_M} \approx 10^{-8}.
$$

**Empirical confirmation** ($f = \sin$, $x = 1$, IEEE double). Measured absolute errors:

| $h$ | forward | central | Richardson (order 4) | complex step |
| :--- | :--- | :--- | :--- | :--- |
| $10^{-3}$ | $4.21\times10^{-4}$ | $9.01\times10^{-8}$ | $3.54\times10^{-14}$ | $9.01\times10^{-8}$ |
| $10^{-5}$ | $4.21\times10^{-6}$ | $1.11\times10^{-11}$ | $1.11\times10^{-11}$ | $9.01\times10^{-12}$ |
| $10^{-8}$ | $2.97\times10^{-9}$ | $2.58\times10^{-9}$ | $4.82\times10^{-9}$ | $0$ |
| $10^{-12}$ | $4.32\times10^{-5}$ | $1.23\times10^{-5}$ | $2.10\times10^{-4}$ | $0$ |
| $10^{-16}$ | $5.40\times10^{-1}$ | $1.48\times10^{-2}$ | $7.25\times10^{-1}$ | $0$ |

The best forward value is $2.97\times10^{-9}$ near $h = 10^{-8} \approx \sqrt{\varepsilon_M}$; the best central value is $1.11\times10^{-11}$ near $h = 10^{-5} \approx \varepsilon_M^{1/3}$; the theoretical predictions $h^{\ast} = 2\sqrt{\varepsilon_M} = 3.0\times10^{-8}$ and $h^{\ast} = (3\varepsilon_M \lvert f \rvert / \lvert f''' \rvert)^{1/3} = 1.0\times10^{-5}$ are both correct to within a factor of $3$. $\blacksquare$

The worst-case bounds overestimate the realized roundoff by about one order of magnitude, because rounding errors have random signs and partially cancel — but the *scaling laws* $\sqrt{\varepsilon_M}$, $\varepsilon_M^{1/3}$, $\varepsilon_M^{1/4}$ are exactly right and are what one designs against.

### Proof 4 — Richardson extrapolation

**Claim.** (Theorem 6, general form.) If $A(h) = A + c_1 h^{p} + c_2 h^{p+q} + O(h^{p+2q})$ with $c_1 \neq 0$, then

$$
R(h) := \frac{2^{p}A(h/2) - A(h)}{2^{p}-1} = A + O(h^{p+q}).
$$

**Proof.** Write the two expansions:

$$
A(h) = A + c_1 h^{p} + c_2 h^{p+q} + \cdots, \qquad A(h/2) = A + \frac{c_1 h^{p}}{2^{p}} + \frac{c_2 h^{p+q}}{2^{p+q}} + \cdots .
$$

Multiply the second by $2^{p}$ and subtract the first:

$$
2^{p}A(h/2) - A(h) = (2^{p} - 1)A + c_1 h^{p}\underbrace{(1 - 1)}_{= 0} + c_2 h^{p+q}\left( 2^{p} 2^{-(p+q)} - 1 \right) + \cdots
$$

The $h^{p}$ term is annihilated exactly. Dividing by $2^{p}-1$:

$$
R(h) = A + c_2 h^{p+q}\,\frac{2^{-q} - 1}{2^{p}-1} + O(h^{p+2q}) = A + O(h^{p+q}). \qquad \blacksquare
$$

**Application to central differences.** Here $A(h) = \frac{f(x+h)-f(x-h)}{2h}$, $p = 2$, $q = 2$, so

$$
R(h) = \frac{4 A(h/2) - A(h)}{3} = f'(x) - \frac{h^{4}}{480}f^{(5)}(x) + O(h^{6}) ,
$$

with the constant obtained from $c_2 = \frac{f^{(5)}}{120}$ and the factor $\frac{2^{-2}-1}{3} = -\frac14$: $-\frac{1}{4}\cdot\frac{h^4}{120} = -\frac{h^4}{480}$.

**Iterating: the derivative Romberg table.** Set $T_{k,0} = A(h/2^{k})$ and

$$
T_{k,m} = \frac{4^{m} T_{k,m-1} - T_{k-1,m-1}}{4^{m}-1},
$$

so column $m$ has order $2m + 2$. The measured error of one extrapolation on $f = \sin$ at $x=1$ is $3.5\times10^{-14}$ at $h = 10^{-3}$ — a **2500-fold** improvement over the plain central difference at the same $h$.

**Where it stops.** Each column subtracts nearly equal numbers, so the roundoff floor rises geometrically: in the table above, the extrapolant is *worse* than the plain central difference for $h \le 10^{-6}$. Practical rule: extrapolate at *moderate* $h$ (where truncation still dominates) and stop after 3–5 columns, monitoring $\lvert T_{k,m} - T_{k,m-1} \rvert$ and halting when it stops decreasing.

**Prerequisite.** The expansion must exist. For a function with a kink, a discontinuous high derivative, or evaluation noise, $A(h)$ has no clean power series and Richardson extrapolates garbage confidently.

### Proof 5 — Deriving any stencil by undetermined coefficients

**Goal.** Find weights $c_j$ such that

$$
\frac{1}{h^{k}}\sum_{j=1}^{m} c_j\, f(x + a_j h) = f^{(k)}(x) + O(h^{p}), \qquad p = m - k .
$$

**Derivation.** Expand each sample about $x$:

$$
f(x + a_j h) = \sum_{i \ge 0} \frac{(a_j h)^{i}}{i!} f^{(i)}(x).
$$

Substitute and collect powers of $h$:

$$
\frac{1}{h^{k}}\sum_j c_j f(x + a_j h) = \sum_{i\ge0} h^{\,i-k} f^{(i)}(x)\, \underbrace{\sum_j c_j \frac{a_j^{\,i}}{i!}}_{=:\,S_i} .
$$

To reproduce $f^{(k)}$ and kill everything below it we require $S_i = \delta_{i,k}$ for $i = 0, \ldots, m-1$ — exactly $m$ linear equations in $m$ unknowns, with the **transposed Vandermonde** coefficient matrix $\left(\frac{a_j^{i}}{i!}\right)_{i,j}$, nonsingular because the $a_j$ are distinct. Hence *every* stencil has a unique weight set, and the leading error term is

$$
h^{\,m-k} f^{(m)}(x)\, S_m = h^{\,p} f^{(m)}(x) \sum_j c_j \frac{a_j^{\,m}}{m!} .
$$

**Worked example — the five-point first derivative.** Take $k=1$, $a = (-2,-1,0,1,2)$, $m = 5$. Symmetry ($f'$ is odd) forces $c_0 = 0$ and $c_{-j} = -c_j$, leaving $c_1, c_2$. The conditions $S_1 = 1$ and $S_3 = 0$ read

$$
2c_1 + 4c_2 = 1, \qquad \frac{2c_1 + 16 c_2}{6} = 0 \implies c_1 = -8c_2 .
$$

Substituting: $-16c_2 + 4c_2 = 1 \Rightarrow c_2 = -\frac{1}{12}$, $c_1 = \frac{8}{12} = \frac{2}{3}$. So

$$
f'(x) \approx \frac{f(x-2h) - 8f(x-h) + 8f(x+h) - f(x+2h)}{12h},
$$

and the leading error is $h^{4}f^{(5)}(x)\,S_5$ with

$$
S_5 = \frac{1}{120}\left[ \tfrac{2}{3}(1)^5 - \tfrac{2}{3}(-1)^5 - \tfrac{1}{12}(2)^5 + \tfrac{1}{12}(-2)^5 \right] = \frac{1}{120}\left[ \tfrac43 - \tfrac{64}{12} \right] = \frac{-4}{120} = -\frac{1}{30},
$$

i.e. error $-\frac{h^4}{30}f^{(5)}(\xi)$, matching Theorem 4. $\blacksquare$

**Remarks.**
- **One-sided stencils** come from the same machinery with $a = (0,1,2,\ldots)$ — essential at domain boundaries, where they are one order less accurate for the same width and have larger error constants.
- **Non-uniform grids** are handled identically: put the actual offsets in $a_j$. This is how finite-difference schemes on stretched meshes are generated.
- **Fornberg's algorithm** (1988) computes these weights for arbitrary nodes and all derivative orders simultaneously in $O(m^2)$ with a stable recursion, avoiding the ill-conditioned Vandermonde solve.
- **Equivalent view**: the stencil is the exact derivative of the interpolating polynomial through the stencil points — differentiating Topic 04's error formula reproduces these error terms.

### Proof 6 — Complex-step differentiation: second order with zero cancellation

**Claim.** (Theorem 7.) For $f$ real-analytic at $x$ (and real on the real axis),

$$
\frac{\operatorname{Im} f(x + ih)}{h} = f'(x) - \frac{h^{2}}{6}f'''(x) + O(h^{4}).
$$

**Proof.** Analyticity gives a convergent Taylor series in the complex step $ih$:

$$
f(x + ih) = f(x) + ih f'(x) + \frac{(ih)^{2}}{2}f''(x) + \frac{(ih)^{3}}{6}f'''(x) + \frac{(ih)^4}{24}f^{(4)}(x) + \cdots
$$

Using $i^2 = -1$, $i^3 = -i$, $i^4 = 1$ and the fact that all $f^{(k)}(x)$ are **real**:

$$
f(x+ih) = \underbrace{\left[ f(x) - \frac{h^{2}}{2}f''(x) + \frac{h^4}{24}f^{(4)}(x) - \cdots \right]}_{\operatorname{Re}} + i\underbrace{\left[ h f'(x) - \frac{h^{3}}{6}f'''(x) + \cdots \right]}_{\operatorname{Im}} .
$$

Dividing the imaginary part by $h$ gives the claim. $\blacksquare$

**Why there is no cancellation.** The imaginary part of $f(x+ih)$ is *itself* of size $O(h)$ — it is not the difference of two $O(1)$ quantities. Nothing nearly-equal is ever subtracted, so the relative accuracy of $\operatorname{Im} f(x+ih)$ is $O(\varepsilon_M)$ regardless of how small $h$ is, and dividing by $h$ preserves relative accuracy exactly. The roundoff term is $O(\varepsilon_M \lvert f' \rvert)$ **independent of $h$**, so the total error is

$$
E(h) \approx \frac{h^{2}}{6}\lvert f''' \rvert + \varepsilon_M \lvert f' \rvert ,
$$

which is monotone decreasing in $h$ down to $h \to 0$: just take $h = 10^{-20}$ and the truncation term is $10^{-40}$, utterly negligible. The measured errors for $f = \sin$ at $x=1$ confirm this exactly: $9.0\times10^{-8}$ at $h = 10^{-3}$ (pure truncation, matching $h^2/6 \cdot \lvert \sin 1 \rvert$), $8.9\times10^{-16}$ at $h = 10^{-7}$, and **exactly $0$** for every $h \le 10^{-8}$.

**Practical caveats.**
1. $f$ must be implemented with complex-capable arithmetic and must be **analytic** — every operation must be the complex extension of the real one. The killers are `abs`, `max`, `min`, and comparison-based branches, plus any use of `real()` inside the code.
2. Functions like $\lvert x \rvert$, $\max(0, x)$ (ReLU), and clipping are not analytic; complex step silently returns a wrong answer rather than failing.
3. The *second* derivative is **not** obtainable this way without cancellation: $\operatorname{Re} f(x+ih) = f(x) - \frac{h^2}{2}f''(x)$ requires subtracting $f(x)$, restoring the problem. (Multicomplex or hyper-dual numbers fix this — and hyper-dual numbers are essentially forward-mode automatic differentiation.)
4. Cost: one complex evaluation $\approx$ 2–4 real evaluations, comparable to forward-mode AD on one input.

## 4. Computational & Algorithmic Insights

### The stencil catalogue

| Target | Formula | Order | Leading error |
| :--- | :--- | :--- | :--- |
| $f'$ forward | $\frac{f(x+h)-f(x)}{h}$ | 1 | $\frac{h}{2}f''$ |
| $f'$ backward | $\frac{f(x)-f(x-h)}{h}$ | 1 | $-\frac{h}{2}f''$ |
| $f'$ central | $\frac{f(x+h)-f(x-h)}{2h}$ | 2 | $\frac{h^2}{6}f'''$ |
| $f'$ one-sided 3-pt | $\frac{-3f(x)+4f(x+h)-f(x+2h)}{2h}$ | 2 | $-\frac{h^2}{3}f'''$ |
| $f'$ 5-point central | $\frac{f(x-2h)-8f(x-h)+8f(x+h)-f(x+2h)}{12h}$ | 4 | $-\frac{h^4}{30}f^{(5)}$ |
| $f''$ central | $\frac{f(x+h)-2f(x)+f(x-h)}{h^2}$ | 2 | $\frac{h^2}{12}f^{(4)}$ |
| $f''$ 5-point | $\frac{-f(x-2h)+16f(x-h)-30f(x)+16f(x+h)-f(x+2h)}{12h^2}$ | 4 | $-\frac{h^4}{90}f^{(6)}$ |
| $f'$ complex step | $\frac{\operatorname{Im} f(x+ih)}{h}$ | 2 | $\frac{h^2}{6}f'''$, no roundoff floor |

Note the one-sided 3-point rule has **twice** the error constant of the central rule of the same order — accuracy at a boundary always costs something.

### Choosing $h$ in practice

Never use an absolute step. Scale it with the magnitude of $x$ so that $x + h$ is a representable perturbation of comparable relative size:

```python
import numpy as np

def central_diff(f, x, h=None):
    eps = np.finfo(float).eps
    if h is None:
        h = eps ** (1 / 3) * max(abs(x), 1.0)      # ~6e-6 * scale
    xp, xm = x + h, x - h
    h_eff = xp - xm                                 # exact representable step
    return (f(xp) - f(xm)) / h_eff
```

The trick `h_eff = xp - xm` matters: $x + h$ is rounded to a machine number, so the *effective* step differs from the nominal $h$ by up to $\varepsilon_M \lvert x \rvert$. Using the exact difference of the perturbed points removes this error entirely — a one-line change worth several digits when $\lvert x \rvert \gg 1$.

### Finite differences versus automatic differentiation

| Property | Finite differences | Forward-mode AD | Reverse-mode AD |
| :--- | :--- | :--- | :--- |
| Accuracy | $O(\sqrt{\varepsilon_M})$ to $O(\varepsilon_M^{2/3})$ | machine precision | machine precision |
| Step size | must be chosen; fragile | none | none |
| Cost of full gradient, $n$ inputs | $n+1$ (forward) or $2n$ (central) evaluations | $n$ sweeps, each $\approx 2\times$ cost of $f$ | **one** sweep, $\approx 3\text{--}4\times$ cost of $f$ |
| Memory | $O(1)$ | $O(1)$ | $O(\text{graph size})$ |
| Needs source access | no (black box) | yes | yes |
| Non-smooth $f$ | wrong at kinks | subgradient convention | subgradient convention |

The decisive fact is the **cheap gradient principle**: reverse-mode AD computes $\nabla f \in \mathbb{R}^{n}$ at a cost of $O(1)$ function evaluations, independent of $n$. For a network with $n = 10^{9}$ parameters, central differences would need $2\times10^{9}$ forward passes per gradient — about $10^{9}$ times slower. This is why backpropagation, not finite differencing, trains neural networks.

Finite differences nonetheless survive in three niches:
1. **Verification** — checking a hand-written or custom-kernel gradient against a method that shares no code with it.
2. **Black boxes** — legacy Fortran simulators, closed-source solvers, hardware-in-the-loop measurements, and any objective evaluated by an experiment.
3. **Matrix-free second-order information** — Hessian-vector products from gradient differences, requiring no second-order AD.

```python
def hessian_vector_product(grad_f, x, v, eps=None):
    # (grad(x + h v) - grad(x - h v)) / (2h) ~= H(x) v, matrix free
    import numpy as np
    if eps is None:
        eps = np.sqrt(np.finfo(float).eps) * (1.0 + np.linalg.norm(x)) / max(np.linalg.norm(v), 1e-300)
    return (grad_f(x + eps * v) - grad_f(x - eps * v)) / (2 * eps)
```

### Differentiating noisy data — do not use small $h$

If $f$ is known only to accuracy $\eta \gg \varepsilon_M$ (measurements, Monte Carlo estimates, an inner solver run to tolerance $10^{-6}$), then the optimal steps rescale immediately:

$$
h^{\ast}_{\text{fwd}} = 2\sqrt{\frac{\eta \lvert f \rvert}{M_2}}, \qquad h^{\ast}_{\text{cen}} = \left( \frac{3\eta \lvert f \rvert}{M_3} \right)^{1/3},
$$

so a Monte Carlo objective with $\eta = 10^{-4}$ needs $h^{\ast}_{\text{cen}} \approx (3\times10^{-4})^{1/3} \approx 0.067$ — an enormous step, and even then the best attainable accuracy is only $\eta^{2/3} \approx 2\times10^{-3}$. Using $h = 10^{-6}$ on such a function returns pure noise amplified by $10^{6}$.

The principled alternatives are **regularized differentiation** methods:
- Fit a local polynomial or smoothing spline (Topic 04) to a window of points and differentiate the fit — this is the Savitzky–Golay filter, whose derivative coefficients come from the same undetermined-coefficients machinery in a least-squares rather than interpolatory setting.
- Tikhonov-regularized differentiation: minimize $\lVert \mathcal{A}g - f \rVert^2 + \alpha \lVert g' \rVert^2$ where $\mathcal{A}$ is integration.
- Spectral differentiation with a low-pass filter, when the data are periodic and densely sampled.

All of them trade a little bias for a large variance reduction — the same bias–variance dial as smoothing splines, applied to an unstable operator.

## 5. Real-World Physics & AI/ML Applications

**Finite-difference PDE solvers.** Every explicit or implicit finite-difference discretization of the heat, wave, or Navier–Stokes equations is built from the stencils above. The classical FTCS heat-equation scheme is a forward difference in $t$ and a central second difference in $x$; the CFL/von Neumann stability condition $\Delta t \le \frac{\Delta x^{2}}{2\alpha}$ arises from requiring the amplification factor $1 - 4r\sin^2(k\Delta x/2)$ (with $r = \alpha\Delta t/\Delta x^2$) to have modulus $\le 1$. High-order compact and spectral schemes are just higher-order stencils, derived by exactly the undetermined-coefficients procedure of Proof 5.

**Sensitivity analysis and design optimization.** Aerodynamic shape optimization, structural design, and circuit tuning all need $\partial(\text{objective})/\partial(\text{design variable})$ for expensive black-box simulators. Historically these came from forward differences at enormous cost and questionable accuracy; the complex-step method (Squire–Trapp 1998, Martins et al. 2003) was adopted precisely because it gives machine-precision sensitivities from a solver merely recompiled with complex arithmetic, and the adjoint (reverse-mode) method superseded it where source modification is feasible.

**Experimental physics.** Velocity from position data, heat capacity $C = \partial U/\partial T$ from calorimetry, and reaction rates from concentration curves are all numerical differentiation of *noisy* data — the regime where small $h$ is catastrophic and Savitzky–Golay or spline smoothing is mandatory.

**Newton's method and Jacobians.** When an analytic Jacobian is unavailable, $J_{ij} \approx \frac{F_i(x + h e_j) - F_i(x)}{h}$ costs $n$ extra residual evaluations. Because the resulting $J$ is only $\sqrt{\varepsilon_M}$-accurate, Newton's quadratic convergence degrades to roughly $1.5$-order — still fast, and the basis of every `scipy.optimize` solver called without a `jac` argument.

**Machine learning applications.**

- **Gradient checking.** The standard correctness test for a hand-written backward pass, a custom CUDA kernel, or a novel layer:
$$
\text{rel. error} = \frac{\lVert \nabla_{\text{analytic}} - \nabla_{\text{FD}} \rVert}{\lVert \nabla_{\text{analytic}} \rVert + \lVert \nabla_{\text{FD}} \rVert} \lt 10^{-6} \ (\text{float64}).
$$
`torch.autograd.gradcheck` uses central differences with $h = 10^{-6}$ and **requires float64** — in float32, $\varepsilon_M \approx 6\times10^{-8}$ makes $\varepsilon_M^{2/3} \approx 1.5\times10^{-5}$ the best possible agreement, so float32 gradient checks produce false alarms. Test at points away from kinks: ReLU, `max`, `abs`, clipping, and quantization all have undefined derivatives where the finite difference straddles the kink and returns the *secant*, not the subgradient.

- **Hessian-vector products.** Newton-CG, trust-region optimizers, K-FAC diagnostics, Lanczos spectrum estimation of the loss surface, and influence functions all need $Hv$ but never the full $H$ (which would be $10^{18}$ entries for a $10^{9}$-parameter model). The finite-difference form $Hv \approx \frac{\nabla f(x + hv) - \nabla f(x - hv)}{2h}$ costs two gradient evaluations, no second-order AD, and no extra memory; the step is chosen as $h = \sqrt{\varepsilon_M}\,(1 + \lVert x \rVert)/\lVert v \rVert$ so that the perturbation is relatively sized. (Exact `hvp` via double-backward AD is preferred when available — it has no step-size error at all.)

- **Zeroth-order and black-box optimization.** Reinforcement learning from a non-differentiable simulator, prompt optimization over a closed API, and hardware-in-the-loop tuning use randomized finite differences: SPSA estimates the gradient with **two** evaluations regardless of dimension via $\hat g_i = \frac{f(x + c\Delta) - f(x - c\Delta)}{2c\Delta_i}$ with random $\pm1$ perturbations $\Delta$; evolution strategies use $\hat g = \frac{1}{\sigma N}\sum_k \epsilon_k f(x + \sigma\epsilon_k)$, a smoothed directional-derivative estimate. Both are finite differencing with the step-size analysis of Proof 3 governing $c$ and $\sigma$.

- **Differentiable simulation interfaces.** When a physics engine or renderer is not differentiable, practitioners either finite-difference it (accepting $\sqrt{\varepsilon_M}$ accuracy) or rewrite it in JAX/PyTorch to get exact gradients. The accuracy gap in Proof 3 is the quantitative argument for paying the rewrite cost.

- **Learning-rate and hyperparameter sensitivity.** Hypergradients $\partial \mathcal{L}_{\text{val}}/\partial \lambda$ for hyperparameters $\lambda$ are often estimated by finite differences over full training runs — the ultimate noisy-$f$ regime ($\eta \sim 10^{-2}$ from seed variance), where Proof 3 says $h$ must be *large* and precision is fundamentally limited.

### Summary of key results

| Result | Statement |
| :--- | :--- |
| Forward difference | $f'(x) + \frac{h}{2}f''(\xi)$, order 1 |
| Central difference | $f'(x) + \frac{h^2}{6}f'''(\xi)$, order 2, even-power expansion |
| Second derivative | $f''(x) + \frac{h^2}{12}f^{(4)}(\xi)$, order 2 |
| Five-point first derivative | error $-\frac{h^4}{30}f^{(5)}(\xi)$, order 4 |
| Total error model | $C_t h^{p} + C_r \varepsilon_M / h$, V-shaped on log-log axes |
| Optimal step (forward) | $h^{\ast} = 2\sqrt{\eta \lvert f \rvert / M_2} \sim \sqrt{\varepsilon_M}$, best error $\sim \sqrt{\varepsilon_M}$ |
| Optimal step (central) | $h^{\ast} = (3\eta \lvert f \rvert / M_3)^{1/3} \sim \varepsilon_M^{1/3}$, best error $\sim \varepsilon_M^{2/3}$ |
| Optimal step ($f''$) | $h^{\ast} \sim \varepsilon_M^{1/4} \approx 1.2\times10^{-4}$, best error $\sim \sqrt{\varepsilon_M}$ |
| Richardson | $\frac{2^{p}A(h/2)-A(h)}{2^{p}-1} = A + O(h^{p+q})$; central differences jump $2 \to 4$ |
| Stencil weights | transposed Vandermonde system $\sum_j c_j a_j^{i}/i! = \delta_{i,k}$ |
| Complex step | $\operatorname{Im} f(x+ih)/h = f' + O(h^2)$, no cancellation, exact at $h = 10^{-20}$ |
| Reverse-mode AD | full gradient in $O(1)$ function evaluations, machine precision |

## 6. Canonical Literature Mapping & References

| Concept | Canonical source | Location |
| :--- | :--- | :--- |
| Difference formulas and Taylor error terms | Burden & Faires, *Numerical Analysis* | Ch. 4.1 |
| Richardson extrapolation | Burden & Faires; Quarteroni et al. | Ch. 4.2; Ch. 10 |
| Roundoff–truncation trade-off, optimal $h$ | Press et al., *Numerical Recipes* | Ch. 5.7 |
| Conditioning and instability of differentiation | Heath, *Scientific Computing* | Ch. 8.6 |
| Arbitrary-node stencil weights | Fornberg, *Math. Comp.* 51 (1988) | entire paper |
| Complex-step derivative | Squire & Trapp, *SIAM Review* 40 (1998); Martins et al., *ACM TOMS* 29 (2003) | entire papers |
| Automatic differentiation theory and cost bounds | Griewank & Walther, *Evaluating Derivatives* | Chs. 3–4 |
| Finite differences for PDEs, stability | Quarteroni, Sacco & Saleri | Ch. 12 |
| Savitzky–Golay and noisy differentiation | Press et al. | Ch. 14.9 |
| Backward error and stability framework | Trefethen & Bau, *Numerical Linear Algebra* | Lectures 12–15 |

**Primary references.** Burden & Faires (Ch. 4.1–4.2); Heath (Ch. 8.6); Press et al. (Ch. 5.7); Fornberg (1988); Squire & Trapp (1998); Martins, Sturdza & Alonso (2003); Griewank & Walther (2008); Quarteroni, Sacco & Saleri (Chs. 10, 12).